# **CLIP-Dissect: describing a channel in words, without annotation**

Practice for the module [«What the network has learned: features and concepts»](https://open-xai-platform.web.app).

Network Dissection needs Broden — a dataset with per-pixel annotation of concepts. You have to
get it from somewhere, and it will only find the concepts that are in it. CLIP-Dissect lifts both
limits: any set of images instead of masks, any list of words instead of an annotation vocabulary.

Here we assemble the method end to end:

- compute an «image × word» similarity matrix with CLIP;
- record the responses of a channel of our network on the same set of images;
- match the two rankings and get a textual description of the channel — **without a single mask**.

In [ ]:
!pip install open_clip_torch -q

In [ ]:
import torch
import torch.nn.functional as F
import requests
from io import BytesIO
from PIL import Image
from torchvision import models, transforms
from torchvision.models import ResNet50_Weights
import open_clip

torch.manual_seed(0);

## 1. What the method needs

Three things, and none of them requires annotation:

| What | What for | Where it comes from here |
| --- | --- | --- |
| a probe set of images $D$ | both the channel responses and the word similarity are measured on it | course images and their crops |
| a vocabulary of concepts $S$ | the description is picked from it | a list of words written by hand |
| the network under study | its channels are what we describe | ResNet-50, layer `layer4` |

In real work $D$ has thousands of images and the vocabulary thousands of words. Here the set is
small so that the notebook runs on CPU in a minute; the logic does not change.

In [ ]:
def load(name):
    return Image.open(BytesIO(requests.get('https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/' + name).content)).convert('RGB')

sources = ['hog.jpg', 'cat.jpg', 'cat_and_dog.jpg', 'pig.png']
originals = [load(n) for n in sources]

# The probe set: the images themselves plus four crops of each — more responses that way,
# and still no annotation needed.
probe = []
for im in originals:
    w, h = im.size
    probe.append(im)
    for box in [(0, 0, w // 2, h // 2), (w // 2, 0, w, h // 2),
                (0, h // 2, w // 2, h), (w // 2, h // 2, w, h)]:
        probe.append(im.crop(box))
print('images in the probe set:', len(probe))

In [ ]:
vocabulary = [
    'a photo of a pig', 'a photo of a cat', 'a photo of a dog',
    'fur texture', 'grass', 'a snout', 'an eye', 'a paw',
    'a wooden fence', 'sky', 'dirt and mud', 'whiskers',
]
print('words in the vocabulary:', len(vocabulary))

## 2. The «image × word» similarity matrix

CLIP puts images and texts into one space, so similarity is the cosine between the embeddings.
Our own network is not involved yet: this is a property of CLIP itself.

In [ ]:
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    'ViT-B-32', pretrained='laion2b_s34b_b79k')
clip_model.eval()
tokenizer = open_clip.get_tokenizer('ViT-B-32')

with torch.no_grad():
    img_emb = clip_model.encode_image(torch.stack([clip_preprocess(im) for im in probe]))
    txt_emb = clip_model.encode_text(tokenizer(vocabulary))
    img_emb = F.normalize(img_emb, dim=-1)
    txt_emb = F.normalize(txt_emb, dim=-1)

similarity = img_emb @ txt_emb.T           # (images, words)
print('similarity matrix:', tuple(similarity.shape))

## 3. Responses of a channel of the network under study

For every image of the probe set we record the mean activation of a channel. That gives a vector
of length «number of images» — one per channel.

In [ ]:
model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
model.eval();

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

store = {}
handle = model.layer4[-1].register_forward_hook(lambda m, i, o: store.__setitem__('a', o))
with torch.no_grad():
    model(torch.stack([transform(im) for im in probe]))
handle.remove()

activations = store['a'].mean(dim=(2, 3))   # (images, channels)
print('activation matrix:', tuple(activations.shape))

## 4. Matching the two rankings

The naive move is a correlation — but, as the authors note, a channel often responds strongly to
only a small share of the images, and a correlation blurs such a profile. The measure is therefore
deliberately asymmetric: what matters is that «striped» pictures excite the channel, not that all
striped pictures do.

We take a simple version of the same idea: look only at the top of the channel activations and ask
which word is on average closest to those images.

In [ ]:
def describe(channel, top_k=5):
    """The vocabulary word that best describes the top activations of the channel."""
    top = torch.topk(activations[:, channel], top_k).indices
    score = similarity[top].mean(0)
    best = score.argmax().item()
    return vocabulary[best], score[best].item()

for channel in (0, 7, 42, 100, 512, 1000, 2047):
    word, score = describe(channel)
    print(f'channel {channel:4d} → «{word}»  (similarity {score:.3f})')

**Task 1.** Add five words of your own to the vocabulary — say `'a wheel'`, `'a window'`,
`'water'`. Have the channel descriptions changed? What does that say about the method dependence
on the vocabulary?

**Task 2.** Find a channel whose top activations consist of crops of different images with
different objects. What description does it get, and why can it not be trusted?

In [ ]:
# Your code here

## 5. What the method does not give

Let us check the main limitation from the lesson by hand: **CLIP-Dissect assigns one concept per
channel**. If the channel is polysemantic, the second and third concepts simply go unnamed.

In [ ]:
channel = 42
top = torch.topk(activations[:, channel], 5).indices
score = similarity[top].mean(0)
order = torch.topk(score, 3)

print(f'channel {channel} — the three closest words, not one:')
for v, i in zip(order.values, order.indices):
    print(f'   «{vocabulary[i]}»  {v:.3f}')
print('\nGap between the first and the second:', round((order.values[0] - order.values[1]).item(), 3))

**Task 3.** If the gap between the first and the second word is small, describing the channel
with one word loses half the picture. Compute that gap for ten arbitrary channels. For how many
of them is it below 0.01?

**Task 4.** The method describes the channel in the words of **another model** — CLIP. Come up
with a way to check that the description is about our network rather than about the blind spots
of CLIP. (The hint from the lesson: show the top-activating images to a human.)

In [ ]:
# Your code here

## What to take away

- **No annotation is needed.** The method compares not regions in an image but two rankings of
  images — «what the channel responds to more» against «what is more similar to the word».
  There are no masks and no threshold $T_k$ here at all.
- **The vocabulary can be changed freely** — and the answer depends on it. It is a hyperparameter
  of the explanation and has to be reported along with the result.
- **One concept per channel.** The method removes the vocabulary problem of Network Dissection,
  but not the «one unit — one concept» problem.
- **The description is produced by another model.** The errors and biases of CLIP land straight
  in our explanation, and the only way to check that is externally.

**Labels:** global, model-agnostic with respect to the network under study — only activations are
needed. Input: a network, any set of images, any vocabulary. Output: a textual description of a channel.